# TomatoLab — 학습 결과 ZIP으로 ONNX 모델 만들기

학습을 다시 할 필요 없이 이전에 다운로드한 **TomatoLab_results.zip** 파일로 브라우저용 ONNX 모델을 만드는 노트북입니다.

1. 아래 셀을 순서대로 실행합니다.
2. 파일 선택창에서 `TomatoLab_results.zip`을 업로드합니다.
3. 변환 후 `resnet18.onnx` 파일을 다운로드합니다.

> ONNX 파일은 수십 MB입니다. 브라우저 분류 데모와 실제 진단은 다릅니다.

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path('/content/TomatoLab')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/oyueo-mm/TomatoLab.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'onnx>=1.16', 'onnxruntime>=1.18'], check=True)
print('모델 변환 준비 완료')

In [ ]:
import io, zipfile
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('TomatoLab_results.zip 파일을 선택해 주세요.')
name, archive = next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    if 'resnet18/best.pt' not in z.namelist():
        raise ValueError('ZIP 안에 resnet18/best.pt 파일이 없습니다.')
    checkpoint = Path('artifacts/resnet18/best.pt')
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    checkpoint.write_bytes(z.read('resnet18/best.pt'))
print('체크포인트 복원 완료:', checkpoint, '(', round(checkpoint.stat().st_size/1024/1024, 1), 'MiB)')

In [ ]:
subprocess.run([sys.executable, 'scripts/export_onnx.py', '--checkpoint', 'artifacts/resnet18/best.pt', '--output', 'docs/models/resnet18.onnx'], check=True)
print('ONNX 변환·출력 검증 완료')
files.download('docs/models/resnet18.onnx')

## 변환 이후

[TomatoLab 웹 UI 폴더](https://github.com/oyueo-mm/TomatoLab/tree/main/docs)에는 .onnx 파일을 선택해 사진을 분류하는 코드가 들어 있습니다.

공개 사이트에서 모든 방문자가 바로 사용하도록 하려면 ONNX 파일을 Git 또는 GitHub Desktop으로 `docs/models/resnet18.onnx`에 푸시하고, `docs/config.json`의 `modelUrl`을 `./models/resnet18.onnx`로 지정해야 합니다.

GitHub Pages도 Settings → Pages에서 **GitHub Actions**로 활성화해야 합니다.